# Notebook 3 – Similarity Analysis

## Mục tiêu

Notebook này phân tích điểm **Cosine Similarity** được tính ở Notebook 2, nhằm quan sát mức độ tương đồng giữa ảnh thử nghiệm và ảnh tham chiếu ở từng điều kiện.

Notebook tập trung vào:
1. Đọc kết quả Similarity đã lưu từ Notebook 2.
2. Thống kê điểm Similarity theo từng điều kiện.
3. Quan sát phân phối điểm và ngưỡng nhận diện `0.7`.
4. Kiểm tra các ảnh có điểm gần ngưỡng.

**Lưu ý:** Notebook này sử dụng lại kết quả từ Notebook 2, không chạy lại MTCNN hoặc FaceNet.

## 1. Chuẩn bị thư viện và đọc dữ liệu

Trước tiên, chúng ta import các thư viện cần thiết và đọc file kết quả được xuất từ Notebook 2.

**Yêu cầu:** Notebook 2 cần lưu `results_df` thành file `results/similarity_results.csv` trước khi chạy Notebook này.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# Tìm file kết quả từ thư mục notebook
results_path = Path("../results/similarity_results.csv")

if not results_path.exists():
    raise FileNotFoundError(
        "Không tìm thấy results/similarity_results.csv. "
        "Hãy chạy Notebook 2 và lưu results_df trước."
    )

results_df = pd.read_csv(results_path)

# Kiểm tra các cột cần thiết
required_columns = {"filename", "condition", "similarity", "prediction"}
missing_columns = required_columns - set(results_df.columns)

if missing_columns:
    raise ValueError(f"File kết quả đang thiếu các cột: {missing_columns}")

# Đảm bảo cột similarity có kiểu số
results_df["similarity"] = pd.to_numeric(
    results_df["similarity"], errors="coerce"
)
results_df = results_df.dropna(subset=["similarity"])

print("Số ảnh có kết quả hợp lệ:", len(results_df))
display(results_df.head())

## 2. Thống kê điểm Similarity theo điều kiện

Phần này tính điểm Similarity trung bình, thấp nhất và cao nhất của từng điều kiện.

- **Mean:** Điểm Similarity trung bình.
- **Min:** Điểm Similarity thấp nhất.
- **Max:** Điểm Similarity cao nhất.

Các giá trị này giúp quan sát điều kiện nào có điểm tương đồng cao hoặc thấp hơn.

In [ ]:
# Thống kê điểm Similarity theo từng điều kiện
similarity_stats = results_df.groupby("condition")["similarity"].agg(
    ["count", "mean", "min", "max"]
)

similarity_stats = similarity_stats.rename(columns={
    "count": "Số ảnh",
    "mean": "Similarity trung bình",
    "min": "Similarity thấp nhất",
    "max": "Similarity cao nhất"
})

display(similarity_stats.round(3))

## 3. Trực quan hóa phân phối Similarity

Biểu đồ Boxplot cho phép so sánh phân phối điểm Similarity giữa các điều kiện.

Đường nét đứt biểu thị ngưỡng nhận diện `0.7` đang được sử dụng trong Notebook 2:
- Điểm từ `0.7` trở lên: Matched.
- Điểm dưới `0.7`: Unknown.

Nếu phần lớn điểm của một điều kiện nằm dưới đường ngưỡng, điều kiện đó có thể gây khó khăn cho hệ thống nhận diện.

In [ ]:
# Ngưỡng nhận diện được sử dụng trong Notebook 2
THRESHOLD = 0.7

plt.figure(figsize=(11, 5))

sns.boxplot(
    data=results_df,
    x="condition",
    y="similarity"
)

plt.axhline(
    y=THRESHOLD,
    color="red",
    linestyle="--",
    label=f"Ngưỡng nhận diện ({THRESHOLD})"
)

plt.title("Phân phối Cosine Similarity theo từng điều kiện")
plt.xlabel("Điều kiện thử nghiệm")
plt.ylabel("Cosine Similarity")
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Kiểm tra các ảnh có điểm gần ngưỡng

Những ảnh có điểm Similarity gần `0.7` là các trường hợp đáng chú ý vì chỉ cần điểm thay đổi một chút, kết quả phân loại có thể chuyển giữa Matched và Unknown.

Ở đây, chúng ta chọn các ảnh có điểm nằm trong khoảng `0.65` đến `0.75` để kiểm tra.

In [ ]:
# Lọc các ảnh có điểm Similarity gần ngưỡng 0.7
borderline_df = results_df[
    results_df["similarity"].between(0.65, 0.75)
].copy()

borderline_df = borderline_df.sort_values("similarity")

print("Số ảnh có Similarity từ 0.65 đến 0.75:", len(borderline_df))

display(
    borderline_df[["filename", "condition", "similarity", "prediction"]]
)

## 5. Nhận xét kết quả

Dựa trên bảng thống kê và biểu đồ, có thể nhận xét:

- Điều kiện có Similarity trung bình cao thường tạo ra embedding gần với ảnh tham chiếu hơn trong tập dữ liệu này.
- Điều kiện có nhiều điểm nằm dưới ngưỡng `0.7` sẽ có nhiều ảnh bị phân loại Unknown.
- Các ảnh gần ngưỡng cần được xem xét thêm vì kết quả nhận diện của chúng nhạy với giá trị Threshold.

**Lưu ý khi kết luận:** Similarity chỉ phản ánh mức độ tương đồng giữa embedding của ảnh thử nghiệm và ảnh tham chiếu. Kết quả trên tập dữ liệu hiện tại chưa đủ để khẳng định khả năng nhận diện người lạ, vì tập thử nghiệm chưa có danh tính khác để đánh giá khả năng từ chối.